# 1. Final results on the held-out test sets

**What this notebook shows.** The answer to the task (`docs/TASK.md`): does the fused model —
three pathology foundation models (UNI2-h, Virchow2, GenBio-PathFM) plus age and sex — beat every
single model on held-out test patients, and how sure are we?

**How the numbers were made (in one paragraph).** The 203 patients were split 5 times (seeds 0–4)
into 162 development and 41 test patients, by patient and stratified by subtype. The agent chose
its designs on seed 0's development patients only. Three designs were then frozen. For each seed,
every model was trained once on the 162 development patients and predicted the 41 test patients
once. So every design has **5 test results**, one per test set.

**Nothing is retrained in sections 1–6**: they read the saved results in `artifacts/final/`.
Section 7 lets you re-run the evaluation yourself (one seed, or all five).

In [ ]:
# Run from anywhere inside the repository: move to its root so every path below works.
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from src.eval import final_view as fv

fv.style()
pd.set_option("display.max_colwidth", 200)
print("Repository:", root.name)

In [ ]:
runs = fv.load_final("artifacts/final")
primary = fv.primary(runs)
print(f"Seeds loaded: {sorted(runs)}   ·   designs: {fv.designs(runs)}   ·   primary design: {primary}")

## 2. The three frozen designs

All three train one tile-level model per foundation model (it reads 1,024 tiles of the slide and
learns which tiles matter), then multiply their predicted probabilities with a small age-and-sex
model. They differ in how each tile-level model is trained:

| Design | Tile-level model | Role |
| --- | --- | --- |
| **H1** | attention pooling + an extra training target (4 grade groups) | **primary**: the agent's own choice |
| H2 | CLAM (attention + pseudo-labelled tiles) + the grade-group target | highest development score |
| C1 | CLAM, without the grade-group target | best design without it |

The primary design was fixed *before* any test set was read, and is never changed afterwards.

In [ ]:
fv.designs_table(runs)

## 3. The required comparison table (TASK.md §5.3), primary design

Every row uses **the same tile-level recipe**, so the table isolates the effect of fusion
(TASK.md: "keep the downstream classifier family and tuning budget identical across rows").

- *Mean ± sd* is over the 5 test sets.
- *[95% CI]* is a bootstrap interval over the 41 patients of seed 0's test set (the headline split).

In [ ]:
fv.task_table(runs, primary)

**For reference: the original rows 1–5** (each model's tiles simply averaged into one vector, then
logistic regression). These are weaker because averaging dilutes the tumour tiles with normal lung
and stroma.

In [ ]:
fv.baseline_table(runs)

## 4. Test AUROC on each of the 5 test sets

One dot per test set, with its 95% interval. Blue: the fused model. Orange: UNI2-h alone under
the same recipe (the strongest single model). Green: the original row 5.

In [ ]:
fv.plot_per_seed(runs, primary); plt.show()

## 5. Does the fused model beat each single model?

For each test set, the fused model's AUROC minus the other row's AUROC on the **same patients**
(a paired comparison). A row counts as *beaten on a seed* only if the 95% interval of that
difference lies entirely above 0 (rule fixed in advance in `conf/final.yaml`).

In [ ]:
fv.plot_gains(runs, primary); plt.show()

In [ ]:
fv.gain_table(runs, primary)

**How to read this.**
- Against the original rows 1–5, the fused model wins clearly on every test set (+0.05 to +0.06).
  That gain comes from the tile-level recipe.
- Against each model **under the same recipe**, the fused model is ahead of Virchow2 and
  GenBio-PathFM on all 5 test sets, but only level with UNI2-h; no interval lies above 0.
  Fusion matches the best single model; it does not clearly beat it.
- Age and sex add about +0.002.

## 6. Failure analysis

**Per class.** Each test set holds only 3–9 patients per class, so these are means over the 5 sets.

In [ ]:
fv.plot_per_class(runs, primary); plt.show()

**Confusion matrix** (all 5 test sets added up; each patient appears once per test set it is in).
Colour is the share of the true class; numbers are patients. The weak spots are **Lepidic**
(mostly called Acinar or Papillary) and **Papillary** (often called Acinar): the low-grade patterns
that differ by how much of the tumour shows each architecture.

In [ ]:
fv.plot_confusion(runs, primary); plt.show()

**Full tables** — every design, every row, per-seed intervals, development vs test and the
sensitivity analysis without the five patients with conflicting labels — are in `docs/results.md`:

In [ ]:
display(Markdown(Path("docs/results.md").read_text()))

## 7. Re-run the evaluation yourself

This trains every model of the three designs again and predicts the test patients again. It needs
the cached tile embeddings (`artifacts/features/`) and a CUDA GPU; our results used an NVIDIA RTX
4090 (other GPUs may differ in the last digits). Results go to `artifacts/final_rerun/`, so the committed results are **never overwritten**,
and the last cell checks that the re-run reproduces them exactly (the training is deterministic).

**Option A — one seed (default, about 13 minutes on one GPU).**

In [ ]:
fv.run_final_eval([0], devices=("cuda:0",), out_dir="artifacts/final_rerun")

**Option B — all five seeds (about 40 minutes on two GPUs).** Remove the `#` to run it. Seeds run
one per GPU at a time (one seed needs about 10 GB of GPU memory at its peak).

In [ ]:
# fv.run_final_eval([0, 1, 2, 3, 4], devices=("cuda:0", "cuda:1"), out_dir="artifacts/final_rerun")

**Check:** the re-run against the committed results (largest difference over every score; 0 means identical).

In [ ]:
rerun = fv.load_final("artifacts/final_rerun")
fv.compare_runs(rerun, runs)